💡 **Environment:** `clamp-analyses`

# Supplementary Figure 6: Projection benchmarks and mechanism recovery

One 180 × 247 mm page. The cells prepare source data, draw the selected panels, and export PDF, PNG, and SVG.

In [1]:
source(here::here("scripts/panels/style.R"))
suppressPackageStartupMessages({library(ggrepel); library(readr); library(dplyr)})
source(here("scripts/archs4/projections/plots.R"))
source(here("scripts/panels/gm_ids.R"))
set.seed(123)
out <- here("output/99_panels/supp6/source_data")
dir.create(out, recursive=TRUE, showWarnings=FALSE)
agg_dir <- here("output/03_model_biology/02_archs4/03_projections/aggregate")
prod_root <- here("output/01_model_building/02_archs4/03_projections")
pathway_file <- function(name) {
  candidates <- c(here("data", "pathways", name), here("../clamp-analyses/data/pathways", name))
  found <- candidates[file.exists(candidates)]
  if (!length(found)) stop("Missing pathway input: ", name)
  found[1]
}

## Cytokine source data

In [2]:
cyt_dataset <- "cyt_ifna_GSE133218"
hits <- fread(file.path(agg_dir, 'lv_pathway_hits_long.csv'))[dataset == cyt_dataset]
setnames(hits, 'p.adjust', 'fdr', skip_absent = TRUE)

excluded_hub_lvs <- c("LV108", "LV786")
hits <- hits[!LV %chin% excluded_hub_lvs]

manual_mechanisms <- list(
  "Interferon alpha response"        = list(pattern = "HALLMARK_INTERFERON_ALPHA_RESPONSE",
                                             category = "molecular_mechanism", target_contrast = "pooled"),
  "IRF/STAT response"                = list(pattern = "HALLMARK_INTERFERON_ALPHA_RESPONSE",
                                             category = "molecular_mechanism", target_contrast = "pooled"),
  "MHC-I antigen presentation"       = list(pattern = "ANTIGEN_PROCESSING_AND_PRESENTATION_BY_MHC_CLASS_I",
                                             category = "molecular_mechanism", target_contrast = "pooled"),
  "ER stress"                        = list(pattern = "ATF4_ACTIVATES|\\bIRE1\\b|\\bATF6\\b|\\bPERK\\b",
                                             category = "molecular_mechanism", target_contrast = "pooled"),
  "Protein modification/degradation" = list(pattern = "PROTEASOME",
                                             category = "molecular_mechanism", target_contrast = "pooled"),
  "Alternative splicing"             = list(pattern = "SPLIC",
                                             category = "molecular_mechanism", target_contrast = "pooled"),
  "PD-L1/HLA-E immune protection"    = list(pattern = "REGULATION_OF_PD_L1_CD274_TRANSCRIPTION|HLA[_-]?E",
                                             category = "molecular_mechanism", target_contrast = "pooled"),
  "Pancreatic beta cells"            = list(pattern = "BETA_CELL", database = "hallmark",
                                             category = "cell_type", target_contrast = NA_character_)
)

mechanism_key <- vapply(manual_mechanisms, function(s) paste(s$pattern, s$database %||% ""), character(1))
search_specs <- manual_mechanisms[!duplicated(mechanism_key)]
names(search_specs) <- mechanism_key[!duplicated(mechanism_key)]

manual_recovery <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) {
  hm <- hits[model == mdl]
  candidates <- rbindlist(lapply(names(search_specs), function(key) {
    spec <- search_specs[[key]]
    d <- hm[grepl(spec$pattern, term, ignore.case = TRUE)]
    if (!is.null(spec$database)) d <- d[database %chin% spec$database]
    d <- d[order(fdr), .SD[1L], by = LV]
    if (!nrow(d)) return(NULL)
    d[, search_key := key][, .(search_key, LV, term, fdr, database)]
  }))
  setorder(candidates, fdr)
  claimed_lv <- character(0)
  claimed_key <- character(0)
  won <- rbindlist(lapply(seq_len(nrow(candidates)), function(i) {
    r <- candidates[i]
    if (r$LV %chin% claimed_lv || r$search_key %chin% claimed_key) return(NULL)
    claimed_lv <<- c(claimed_lv, r$LV)
    claimed_key <<- c(claimed_key, r$search_key)
    r
  }))
  by_key <- if (nrow(won)) split(won, won$search_key) else list()
  rbindlist(lapply(names(manual_mechanisms), function(nm) {
    key <- mechanism_key[[nm]]
    r <- by_key[[key]]
    spec <- manual_mechanisms[[nm]]
    if (is.null(r)) {
      data.table(mechanism = nm, LV = NA_character_, term = NA_character_,
                 fdr = NA_real_, database = NA_character_, model = mdl,
                 category = spec$category, target_contrast = spec$target_contrast)
    } else {
      data.table(mechanism = nm, LV = r$LV, term = r$term, fdr = r$fdr,
                 database = r$database, model = mdl,
                 category = spec$category, target_contrast = spec$target_contrast)
    }
  }))
}))
manual_recovery[, recovered := !is.na(LV)]
manual_recovery[, mechanism := factor(mechanism, levels = names(manual_mechanisms))]
setorder(manual_recovery, mechanism, model)

print(manual_recovery[, .(mechanism, model, LV, pathway = term, fdr, recovered)])

source(here("scripts", "archs4", "common.R"))

read_gmt_sets <- function(path) {
  x <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
  out <- lapply(x, function(row) unique(row[-c(1L, 2L)]))
  names(out) <- vapply(x, `[[`, "", 1L)
  out
}
gene_sets <- list(
  canonical = read_gmt_sets(pathway_file("c2.cp.v2026.1.Hs.symbols.gmt")),
  hallmark  = read_gmt_sets(pathway_file("h.all.v2026.1.Hs.symbols.gmt"))
)

z_registry <- fread(file.path(prod_root, cyt_dataset, "mechanism_models.tsv"))
z_mats <- list(
  ARCHS4 = read_matrix_csv(here("output", "98_final_models", "clampfull", "canonical", "archs4", "Z.csv")),
  local  = read_matrix_csv(here(z_registry[model == "local"]$z))
)

top_pct <- 0.01
recovered_rows <- manual_recovery[recovered == TRUE]

gene_loadings <- rbindlist(lapply(seq_len(nrow(recovered_rows)), function(i) {
  r <- recovered_rows[i]
  z <- z_mats[[r$model]]
  members <- intersect(gene_sets[[r$database]][[r$term]], rownames(z))
  n_top <- max(1L, ceiling(nrow(z) * top_pct))
  ord <- order(z[, r$LV], decreasing = TRUE)[seq_len(n_top)]
  data.table(dataset = cyt_dataset, comparison_id = as.character(r$mechanism), model = r$model,
             rank = seq_along(ord), gene = rownames(z)[ord], loading = z[ord, r$LV],
             is_gene_set = rownames(z)[ord] %chin% members, in_top_loading_set = TRUE,
             n_gene_set_in_universe = length(members))
}))

comparisons <- copy(manual_recovery)
setnames(comparisons, "term", "gene_set")
comparisons[, `:=`(dataset = cyt_dataset, comparison_id = as.character(mechanism), top_pct = top_pct)]
comparisons <- merge(
  comparisons,
  gene_loadings[, .(n_gene_set_in_top_loading = sum(is_gene_set),
                     n_gene_set_in_universe = max(n_gene_set_in_universe)),
                by = .(comparison_id, model)],
  by = c("comparison_id", "model"), all.x = TRUE)
comparisons[is.na(n_gene_set_in_top_loading), n_gene_set_in_top_loading := 0L]
comparisons[is.na(n_gene_set_in_universe), n_gene_set_in_universe := 0L]

comparison_tests <- comparisons[, {
  a <- .SD[model == "ARCHS4"]
  b <- .SD[model == "local"]
  p <- NA_real_
  arch_fraction <- NA_real_
  local_fraction <- NA_real_
  if (nrow(a) == 1L && nrow(b) == 1L && isTRUE(a$recovered) && isTRUE(b$recovered) &&
      identical(a$gene_set, b$gene_set) &&
      a$n_gene_set_in_universe > 0L && b$n_gene_set_in_universe > 0L) {
    arch_fraction <- a$n_gene_set_in_top_loading / a$n_gene_set_in_universe
    local_fraction <- b$n_gene_set_in_top_loading / b$n_gene_set_in_universe
    p <- fisher.test(matrix(c(a$n_gene_set_in_top_loading,
                              a$n_gene_set_in_universe - a$n_gene_set_in_top_loading,
                              b$n_gene_set_in_top_loading,
                              b$n_gene_set_in_universe - b$n_gene_set_in_top_loading),
                            nrow = 2, byrow = TRUE), alternative = "greater")$p.value
  }
  .(p_value = p, arch_fraction = arch_fraction, local_fraction = local_fraction)
}, by = comparison_id]
comparison_tests[, p_adj := p.adjust(p_value, method = "BH")]
comparisons <- merge(comparisons, comparison_tests, by = "comparison_id", all.x = TRUE)

comparisons[, comparison_note := ""]
comparisons[model == "local" & !recovered, comparison_note := "Not recovered"]
comparisons[recovered == TRUE & is.na(p_adj), comparison_note := "Different matched pathway"]
comparisons[recovered == TRUE & !is.na(p_adj), comparison_note := paste0(
  fifelse(arch_fraction > local_fraction, "ARCHS4 > Cyt model",
          fifelse(arch_fraction < local_fraction, "Cyt model > ARCHS4", "equal fraction")),
  " (FDR ", formatC(p_adj, format = "e", digits = 1), ")")]

for (name in c("manual_recovery", "gene_loadings", "comparisons"))
  fwrite(get(name), file.path(out, paste0("00_cytokines_", name, ".csv")))
rm(z_mats); gc(verbose=FALSE)

                           mechanism  model     LV
                              <fctr> <char> <char>
 1:        Interferon alpha response ARCHS4   LV66
 2:        Interferon alpha response  local   LV23
 3:                IRF/STAT response ARCHS4   LV66
 4:                IRF/STAT response  local   LV23
 5:       MHC-I antigen presentation ARCHS4  LV882
 6:       MHC-I antigen presentation  local   LV29
 7:                        ER stress ARCHS4  LV134
 8:                        ER stress  local   <NA>
 9: Protein modification/degradation ARCHS4 LV1269
10: Protein modification/degradation  local   <NA>
11:             Alternative splicing ARCHS4   LV49
12:             Alternative splicing  local   <NA>
13:    PD-L1/HLA-E immune protection ARCHS4    LV8
14:    PD-L1/HLA-E immune protection  local    LV9
15:            Pancreatic beta cells ARCHS4   LV63
16:            Pancreatic beta cells  local   LV13
                                                                                  

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1258620,67.3,2390954,127.7,2390954,127.7
Vcells,2884271,22.1,111838976,853.3,118177119,901.7


## Monocyte source data

In [3]:
mono_dataset <- "mono_lps_GSE193336"
hits <- fread(file.path(agg_dir, 'lv_pathway_hits_long.csv'))[dataset == mono_dataset]
setnames(hits, 'p.adjust', 'fdr', skip_absent = TRUE)

excluded_hub_lvs <- c("LV1152", "LV1206")
hits <- hits[!LV %chin% excluded_hub_lvs]

manual_mechanisms <- list(
  "NAD+ depletion / SIRT1 inhibition"                       = list(pattern = "NAD_BIOSYNTHESIS|NAD_METABOLISM|SIRTUIN",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "Increased glucose transport / glycolytic reprogramming"  = list(pattern = "GLYCOLYSIS|GLUCOSE_METABOLISM|GLUCOSE_TRANSPORT",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "Reduced TCA cycle / OXPHOS"                               = list(pattern = "OXIDATIVE_PHOSPHORYLATION|TCA_CYCLE|CITRIC_ACID_CYCLE|RESPIRATORY_ELECTRON_TRANSPORT|\\bOXPHOS\\b",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "Oxidative stress / ROS"                                   = list(pattern = "OXIDATIVE_STRESS|REACTIVE_OXYGEN|ROS_AND_RNS|KEAP1|NFE2L2|NRF2",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "HIF1A / NF-κB inflammatory signaling"                     = list(pattern = "HIF1|NF_KB|NFKB|TOLL_LIKE|TLR4|\\bTNF\\b|INFLAMMASOME",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "Itaconate / macrophage metabolic rewiring"                = list(pattern = "ITACONATE|IRG1|ACOD1",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "Glutathione / redox metabolism"                           = list(pattern = "GLUTATHIONE",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "Neutrophil activation"                                    = list(pattern = "NEUTROPHIL_DEGRANULATION|NEUTROPHIL_ACTIVATION",
                                                                     database = "canonical",
                                                                     category = "molecular_mechanism", target_contrast = "pooled"),
  "Macrophage"                                                = list(pattern = "Macrophage", database = "cellmarker",
                                                                     category = "cell_type", target_contrast = NA_character_),
  "Neutrophil"                                                = list(pattern = "Neutrophil", database = "cellmarker",
                                                                     category = "cell_type", target_contrast = NA_character_)
)

mechanism_key <- vapply(manual_mechanisms, function(s) paste(s$pattern, s$database %||% ""), character(1))
search_specs <- manual_mechanisms[!duplicated(mechanism_key)]
names(search_specs) <- mechanism_key[!duplicated(mechanism_key)]

manual_recovery <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) {
  hm <- hits[model == mdl]
  candidates <- rbindlist(lapply(names(search_specs), function(key) {
    spec <- search_specs[[key]]
    d <- hm[grepl(spec$pattern, term, ignore.case = TRUE)]
    if (!is.null(spec$database)) d <- d[database %chin% spec$database]
    d <- d[order(fdr), .SD[1L], by = LV]
    if (!nrow(d)) return(NULL)
    d[, search_key := key][, .(search_key, LV, term, fdr, database)]
  }))
  setorder(candidates, fdr)
  claimed_lv <- character(0)
  claimed_key <- character(0)
  won <- rbindlist(lapply(seq_len(nrow(candidates)), function(i) {
    r <- candidates[i]
    if (r$LV %chin% claimed_lv || r$search_key %chin% claimed_key) return(NULL)
    claimed_lv <<- c(claimed_lv, r$LV)
    claimed_key <<- c(claimed_key, r$search_key)
    r
  }))
  by_key <- if (nrow(won)) split(won, won$search_key) else list()
  rbindlist(lapply(names(manual_mechanisms), function(nm) {
    key <- mechanism_key[[nm]]
    r <- by_key[[key]]
    spec <- manual_mechanisms[[nm]]
    if (is.null(r)) {
      data.table(mechanism = nm, LV = NA_character_, term = NA_character_,
                 fdr = NA_real_, database = NA_character_, model = mdl,
                 category = spec$category, target_contrast = spec$target_contrast)
    } else {
      data.table(mechanism = nm, LV = r$LV, term = r$term, fdr = r$fdr,
                 database = r$database, model = mdl,
                 category = spec$category, target_contrast = spec$target_contrast)
    }
  }))
}))
manual_recovery[, recovered := !is.na(LV)]
manual_recovery[, mechanism := factor(mechanism, levels = names(manual_mechanisms))]
setorder(manual_recovery, mechanism, model)

print(manual_recovery[, .(mechanism, model, LV, pathway = term, fdr, recovered)])
source(here("scripts", "archs4", "common.R"))

read_gmt_sets <- function(path) {
  x <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
  out <- lapply(x, function(row) unique(row[-c(1L, 2L)]))
  names(out) <- vapply(x, `[[`, "", 1L)
  out
}
read_cellmarker_sets <- function(path, sheet, term_col, gene_col) {
  x <- data.table::as.data.table(readxl::read_excel(path, sheet = sheet))
  x <- x[!is.na(get(term_col)) & !is.na(get(gene_col)),
         .(term = as.character(get(term_col)), gene = as.character(get(gene_col)))]
  split(x$gene, x$term)
}
gene_sets <- list(
  canonical  = read_gmt_sets(pathway_file("c2.cp.v2026.1.Hs.symbols.gmt")),
  hallmark   = read_gmt_sets(pathway_file("h.all.v2026.1.Hs.symbols.gmt")),
  cellmarker = read_cellmarker_sets(pathway_file("Cell_marker_Human.xlsx"),
                                     "human", "cell_name", "Symbol")
)

z_registry <- fread(file.path(prod_root, mono_dataset, "mechanism_models.tsv"))
z_mats <- list(
  ARCHS4 = read_matrix_csv(here("output", "98_final_models", "clampfull", "canonical", "archs4", "Z.csv")),
  local  = read_matrix_csv(here(z_registry[model == "local"]$z))
)

top_pct <- 0.01
recovered_rows <- manual_recovery[recovered == TRUE]

gene_loadings <- rbindlist(lapply(seq_len(nrow(recovered_rows)), function(i) {
  r <- recovered_rows[i]
  z <- z_mats[[r$model]]
  members <- intersect(gene_sets[[r$database]][[r$term]], rownames(z))
  n_top <- max(1L, ceiling(nrow(z) * top_pct))
  ord <- order(z[, r$LV], decreasing = TRUE)[seq_len(n_top)]
  data.table(dataset = mono_dataset, comparison_id = as.character(r$mechanism), model = r$model,
             rank = seq_along(ord), gene = rownames(z)[ord], loading = z[ord, r$LV],
             is_gene_set = rownames(z)[ord] %chin% members, in_top_loading_set = TRUE,
             n_gene_set_in_universe = length(members))
}))

comparisons <- copy(manual_recovery)
setnames(comparisons, "term", "gene_set")
comparisons[, `:=`(dataset = mono_dataset, comparison_id = as.character(mechanism), top_pct = top_pct)]
comparisons <- merge(
  comparisons,
  gene_loadings[, .(n_gene_set_in_top_loading = sum(is_gene_set),
                     n_gene_set_in_universe = max(n_gene_set_in_universe)),
                by = .(comparison_id, model)],
  by = c("comparison_id", "model"), all.x = TRUE)
comparisons[is.na(n_gene_set_in_top_loading), n_gene_set_in_top_loading := 0L]
comparisons[is.na(n_gene_set_in_universe), n_gene_set_in_universe := 0L]

comparison_tests <- comparisons[, {
  a <- .SD[model == "ARCHS4"]
  b <- .SD[model == "local"]
  p <- NA_real_
  arch_fraction <- NA_real_
  local_fraction <- NA_real_
  if (nrow(a) == 1L && nrow(b) == 1L && isTRUE(a$recovered) && isTRUE(b$recovered) &&
      identical(a$gene_set, b$gene_set) &&
      a$n_gene_set_in_universe > 0L && b$n_gene_set_in_universe > 0L) {
    arch_fraction <- a$n_gene_set_in_top_loading / a$n_gene_set_in_universe
    local_fraction <- b$n_gene_set_in_top_loading / b$n_gene_set_in_universe
    p <- fisher.test(matrix(c(a$n_gene_set_in_top_loading,
                              a$n_gene_set_in_universe - a$n_gene_set_in_top_loading,
                              b$n_gene_set_in_top_loading,
                              b$n_gene_set_in_universe - b$n_gene_set_in_top_loading),
                            nrow = 2, byrow = TRUE), alternative = "greater")$p.value
  }
  .(p_value = p, arch_fraction = arch_fraction, local_fraction = local_fraction)
}, by = comparison_id]
comparison_tests[, p_adj := p.adjust(p_value, method = "BH")]
comparisons <- merge(comparisons, comparison_tests, by = "comparison_id", all.x = TRUE)

comparisons[, comparison_note := ""]
comparisons[model == "local" & !recovered, comparison_note := "Not recovered"]
comparisons[recovered == TRUE & is.na(p_adj), comparison_note := "Different matched pathway"]
comparisons[recovered == TRUE & !is.na(p_adj), comparison_note := paste0(
  fifelse(arch_fraction > local_fraction, "ARCHS4 > Monocyte model",
          fifelse(arch_fraction < local_fraction, "Monocyte model > ARCHS4", "equal fraction")),
  " (FDR ", formatC(p_adj, format = "e", digits = 1), ")")]

for (name in c("manual_recovery", "gene_loadings", "comparisons"))
  fwrite(get(name), file.path(out, paste0("01_monocyte_", name, ".csv")))
rm(z_mats); gc(verbose=FALSE)

                                                 mechanism  model     LV
                                                    <fctr> <char> <char>
 1:                      NAD+ depletion / SIRT1 inhibition ARCHS4   <NA>
 2:                      NAD+ depletion / SIRT1 inhibition  local   <NA>
 3: Increased glucose transport / glycolytic reprogramming ARCHS4 LV1457
 4: Increased glucose transport / glycolytic reprogramming  local    LV7
 5:                             Reduced TCA cycle / OXPHOS ARCHS4  LV345
 6:                             Reduced TCA cycle / OXPHOS  local    LV5
 7:                                 Oxidative stress / ROS ARCHS4  LV302
 8:                                 Oxidative stress / ROS  local   <NA>
 9:                   HIF1A / NF-κB inflammatory signaling ARCHS4 LV1269
10:                   HIF1A / NF-κB inflammatory signaling  local   <NA>
11:              Itaconate / macrophage metabolic rewiring ARCHS4   <NA>
12:              Itaconate / macrophage metabolic r

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1281772,68.5,2390954,127.7,2390954,127.7
Vcells,3018782,23.1,112025288,854.7,118177119,901.7


## Placenta source data

In [4]:
placenta_dataset <- "placenta_EMTAB6701"

ora_dbs <- c("canonical", "hallmark", "cellmarker", "azimuth")
hits <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) rbindlist(lapply(ora_dbs, function(db) {
  f <- file.path(prod_root, placenta_dataset, "ora", mdl, db, "enrichment.csv.gz")
  if (!file.exists(f)) return(NULL)
  d <- fread(f)
  if (!nrow(d)) return(NULL)
  d[, `:=`(model = mdl, database = db)]
  d
}))))
setnames(hits, c("ID", "p.adjust"), c("term", "fdr"), skip_absent = TRUE)
hits <- hits[fdr < 0.05 & Count >= 3]

manual_mechanisms <- list(
  "EVT invasion / EMT" =
    list(pattern = "EPITHELIAL_MESENCHYMAL_TRANSITION",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Spiral-artery remodelling" =
    list(pattern = "CELL_SURFACE_INTERACTIONS_AT_THE_VASCULAR_WALL",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "dNK-trophoblast HLA-C / HLA-E / HLA-G interactions" =
    list(pattern = "HLAC_ALLOTYPES_INTERACTIONS_WITH_KIR_ON_DNK_CELLS",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "dNK chemokine / immunomodulatory signaling" =
    list(pattern = "CHEMOKINE_SIGNALING",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Immune checkpoint / maternal-fetal immune tolerance" =
    list(pattern = "CO_INHIBITION_BY_PD_1|IMMUNOREGULATORY_INTERACTIONS_BETWEEN_A_LYMPHOID_AND_A_NON_LYMPHOID_CELL",
         exclude_lv = c("LV1269", "LV948", "LV90"),
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Adenosine-mediated immunoregulation" =
    list(pattern = "ADORA2B",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "dNK1 glycolytic metabolic priming" =
    list(pattern = "KEGG_MEDICUS_REFERENCE_GLYCOLYSIS",
         category = "molecular_mechanism", target_contrast = "trophoblast_vs_other"),
  "Trophoblast" =
    list(pattern = "trophoblast", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Extravillous trophoblast (EVT)" =
    list(pattern = "Extravillous Trophoblasts", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Syncytiotrophoblast (SCT)" =
    list(pattern = "Syncytiotrophoblasts And Villous Cytotrophoblasts", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Villous cytotrophoblast (VCT)" =
    list(pattern = "Syncytiotrophoblasts And Villous Cytotrophoblasts", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Decidual NK / dNK" =
    list(pattern = "UTERINE_NATURAL_KILLER", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Decidual stromal cells" =
    list(pattern = "Stromal", prefer_cellmarker = TRUE, exclude_lv = c("LV90", "LV126"),
         category = "cell_type", target_contrast = "trophoblast_vs_other"),
  "Decidual macrophages / maternal myeloid cells" =
    list(pattern = "Macrophage", prefer_cellmarker = TRUE,
         category = "cell_type", target_contrast = "trophoblast_vs_other")
)

find_best <- function(hm, spec) {
  d <- hm[grepl(spec$pattern, term, ignore.case = TRUE)]
  if (!is.null(spec$exclude_lv)) d <- d[!LV %chin% spec$exclude_lv]
  if (!nrow(d)) return(NULL)
  if (isTRUE(spec$prefer_cellmarker)) {
    d[, rnk := fifelse(database == "cellmarker", 0L, 1L)]
    setorder(d, rnk, fdr)
    d[, rnk := NULL]
  } else setorder(d, fdr)
  d[1]
}

manual_recovery <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) {
  hm <- hits[model == mdl]
  rbindlist(lapply(names(manual_mechanisms), function(nm) {
    spec <- manual_mechanisms[[nm]]
    r <- find_best(hm, spec)
    if (is.null(r)) {
      data.table(mechanism = nm, LV = NA_character_, term = NA_character_, fdr = NA_real_,
                 database = NA_character_, model = mdl, category = spec$category,
                 target_contrast = spec$target_contrast)
    } else {
      data.table(mechanism = nm, LV = r$LV, term = r$term, fdr = r$fdr,
                 database = r$database, model = mdl, category = spec$category,
                 target_contrast = spec$target_contrast)
    }
  }))
}))
manual_recovery[, LV_label := LV]

combined <- rbindlist(lapply(c("local", "ARCHS4"), function(mdl) {
  both <- manual_recovery[model == mdl & mechanism %chin% c("Extravillous trophoblast (EVT)", "Syncytiotrophoblast (SCT)") & !is.na(LV)]
  if (!nrow(both)) {
    return(data.table(mechanism = "Trophoblast differentiation into EVT / SCT", LV = NA_character_,
                       LV_label = NA_character_, term = NA_character_, fdr = NA_real_,
                       database = NA_character_, model = mdl, category = "molecular_mechanism",
                       target_contrast = "trophoblast_vs_other"))
  }
  setorder(both, fdr)
  primary <- both[1]
  data.table(mechanism = "Trophoblast differentiation into EVT / SCT", LV = primary$LV,
             LV_label = paste(unique(both$LV), collapse = " / "),
             term = paste(sort(unique(both$term)), collapse = " / "),
             fdr = primary$fdr, database = primary$database, model = mdl,
             category = "molecular_mechanism", target_contrast = "trophoblast_vs_other")
}))
manual_recovery <- rbind(manual_recovery, combined)

row_order <- c(
  "Trophoblast differentiation into EVT / SCT", "EVT invasion / EMT", "Spiral-artery remodelling",
  "dNK-trophoblast HLA-C / HLA-E / HLA-G interactions", "dNK chemokine / immunomodulatory signaling",
  "Immune checkpoint / maternal-fetal immune tolerance", "Adenosine-mediated immunoregulation",
  "dNK1 glycolytic metabolic priming", "Trophoblast", "Extravillous trophoblast (EVT)",
  "Syncytiotrophoblast (SCT)", "Villous cytotrophoblast (VCT)", "Decidual NK / dNK",
  "Decidual stromal cells", "Decidual macrophages / maternal myeloid cells")
manual_recovery[, recovered := !is.na(LV)]
manual_recovery[, mechanism := factor(mechanism, levels = row_order)]
setorder(manual_recovery, mechanism, model)

print(manual_recovery[, .(mechanism, model, LV_label, pathway = term, fdr, recovered)])

source(here("scripts", "archs4", "common.R"))

read_gmt_sets <- function(path) {
  x <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
  out <- lapply(x, function(row) unique(row[-c(1L, 2L)]))
  names(out) <- vapply(x, `[[`, "", 1L)
  out
}
read_cellmarker_sets <- function(path, sheet, term_col, gene_col) {
  x <- data.table::as.data.table(readxl::read_excel(path, sheet = sheet))
  x <- x[!is.na(get(term_col)) & !is.na(get(gene_col)),
         .(term = as.character(get(term_col)), gene = as.character(get(gene_col)))]
  split(x$gene, x$term)
}
gene_sets <- list(
  canonical  = read_gmt_sets(pathway_file("c2.cp.v2026.1.Hs.symbols.gmt")),
  hallmark   = read_gmt_sets(pathway_file("h.all.v2026.1.Hs.symbols.gmt")),
  azimuth    = read_gmt_sets(pathway_file("Azimuth_2023.txt")),
  cellmarker = read_cellmarker_sets(pathway_file("Cell_marker_Human.xlsx"),
                                     "human", "cell_name", "Symbol")
)

z_registry <- fread(file.path(prod_root, placenta_dataset, "mechanism_models.tsv"))
z_mats <- list(
  ARCHS4 = read_matrix_csv(here("output", "98_final_models", "clampfull", "canonical", "archs4", "Z.csv")),
  local  = read_matrix_csv(here(z_registry[model == "local"]$z))
)

top_pct <- 0.01
recovered_rows <- manual_recovery[recovered == TRUE]

gene_loadings <- rbindlist(lapply(seq_len(nrow(recovered_rows)), function(i) {
  r <- recovered_rows[i]
  z <- z_mats[[r$model]]
  members <- intersect(unique(unlist(gene_sets[[r$database]][strsplit(r$term, " / ", fixed = TRUE)[[1]]])), rownames(z))
  n_top <- max(1L, ceiling(nrow(z) * top_pct))
  ord <- order(z[, r$LV], decreasing = TRUE)[seq_len(n_top)]
  data.table(dataset = placenta_dataset, comparison_id = as.character(r$mechanism), model = r$model,
             rank = seq_along(ord), gene = rownames(z)[ord], loading = z[ord, r$LV],
             is_gene_set = rownames(z)[ord] %chin% members, in_top_loading_set = TRUE,
             n_gene_set_in_universe = length(members))
}))

comparisons <- copy(manual_recovery)
setnames(comparisons, "term", "gene_set")
comparisons[, `:=`(dataset = placenta_dataset, comparison_id = as.character(mechanism), top_pct = top_pct)]
comparisons <- merge(
  comparisons,
  gene_loadings[, .(n_gene_set_in_top_loading = sum(is_gene_set),
                     n_gene_set_in_universe = max(n_gene_set_in_universe)),
                by = .(comparison_id, model)],
  by = c("comparison_id", "model"), all.x = TRUE)
comparisons[is.na(n_gene_set_in_top_loading), n_gene_set_in_top_loading := 0L]
comparisons[is.na(n_gene_set_in_universe), n_gene_set_in_universe := 0L]

comparison_tests <- comparisons[, {
  a <- .SD[model == "ARCHS4"]
  b <- .SD[model == "local"]
  p <- NA_real_
  arch_fraction <- NA_real_
  local_fraction <- NA_real_
  if (nrow(a) == 1L && nrow(b) == 1L && isTRUE(a$recovered) && isTRUE(b$recovered) &&
      identical(a$gene_set, b$gene_set) &&
      a$n_gene_set_in_universe > 0L && b$n_gene_set_in_universe > 0L) {
    arch_fraction <- a$n_gene_set_in_top_loading / a$n_gene_set_in_universe
    local_fraction <- b$n_gene_set_in_top_loading / b$n_gene_set_in_universe
    p <- fisher.test(matrix(c(a$n_gene_set_in_top_loading,
                              a$n_gene_set_in_universe - a$n_gene_set_in_top_loading,
                              b$n_gene_set_in_top_loading,
                              b$n_gene_set_in_universe - b$n_gene_set_in_top_loading),
                            nrow = 2, byrow = TRUE), alternative = "greater")$p.value
  }
  .(p_value = p, arch_fraction = arch_fraction, local_fraction = local_fraction)
}, by = comparison_id]
comparison_tests[, p_adj := p.adjust(p_value, method = "BH")]
comparisons <- merge(comparisons, comparison_tests, by = "comparison_id", all.x = TRUE)

comparisons[, comparison_note := ""]
comparisons[model == "local" & !recovered, comparison_note := "Not recovered"]
comparisons[recovered == TRUE & is.na(p_adj), comparison_note := "Different matched pathway"]
comparisons[recovered == TRUE & !is.na(p_adj), comparison_note := paste0(
  fifelse(arch_fraction > local_fraction, "ARCHS4 > Placenta model",
          fifelse(arch_fraction < local_fraction, "Placenta model > ARCHS4", "equal fraction")),
  " (FDR ", formatC(p_adj, format = "e", digits = 1), ")")]

for (name in c("manual_recovery", "gene_loadings", "comparisons"))
  fwrite(get(name), file.path(out, paste0("02_placenta_", name, ".csv")))
rm(z_mats); gc(verbose=FALSE)

                                              mechanism  model LV_label
                                                 <fctr> <char>   <char>
 1:          Trophoblast differentiation into EVT / SCT ARCHS4     <NA>
 2:          Trophoblast differentiation into EVT / SCT  local     <NA>
 3:                                  EVT invasion / EMT ARCHS4   LV1086
 4:                                  EVT invasion / EMT  local     <NA>
 5:                           Spiral-artery remodelling ARCHS4    LV883
 6:                           Spiral-artery remodelling  local     LV31
 7:  dNK-trophoblast HLA-C / HLA-E / HLA-G interactions ARCHS4    LV883
 8:  dNK-trophoblast HLA-C / HLA-E / HLA-G interactions  local     LV31
 9:          dNK chemokine / immunomodulatory signaling ARCHS4    LV883
10:          dNK chemokine / immunomodulatory signaling  local     LV31
11: Immune checkpoint / maternal-fetal immune tolerance ARCHS4   LV1573
12: Immune checkpoint / maternal-fetal immune tolerance  local  

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1326493,70.9,2390954,127.7,2390954,127.7
Vcells,3084135,23.6,112045888,854.9,119539687,912.1


## Benchmark source data

In [5]:
# Updated benchmark sources are generated by the projection-only Snakemake workflow.


In [6]:
# Updated benchmark sources are generated by the projection-only Snakemake workflow.


In [7]:
file.copy(file.path(agg_dir,"lv_stats_long.csv"), file.path(out,"lv_stats_long.csv"), overwrite=TRUE)
stopifnot(file.exists(file.path(out,"lv_stats_long.csv")))

[1] TRUE

## Assemble and export the figure

In [8]:
# These inputs are regenerated from the current main-repository notebooks by
# export_projection_data.R; they retain the curated mechanisms and contrasts.
out <- if (nzchar(Sys.getenv("SUPP6_SOURCE_DIR"))) Sys.getenv("SUPP6_SOURCE_DIR") else here("output/99_panels/supp6/source_data")
groups <- c("00_cytokines", "01_monocyte", "02_placenta")
S <- lapply(groups, function(g) fread_gm(file.path(out, paste0(g, "_comparisons.csv"))))
L <- lapply(groups, function(g) fread_gm(file.path(out, paste0(g, "_gene_loadings.csv"))))
stats <- fread_gm(file.path(out, "lv_stats_long.csv"))

# A: projection benchmarks with matching-aware Pearson excess and random-LV sensitivity.
source(here("scripts/archs4/projections/benchmark_panels.R"))
benchmark_root <- Sys.getenv("PROJECTION_BENCHMARK_ROOT", unset=here("output/03_model_biology/02_archs4/03_projections/06_dimensionality_adjusted"))
A <- projection_benchmark_panels(benchmark_root)

# B-D: the Figure 3 projection-panel treatment. Local and ARCHS4 use separate
# log2FC scales because the model ranges differ; point size is −log10(ORA FDR).
effect_rows <- lapply(seq_along(S), function(i) {
  s <- copy(S[[i]]); s[, ora_fdr:=fdr]
  rbindlist(lapply(c("local", "ARCHS4"), function(m) {
    p <- proj_expected_mechanism_panel(stats, s, unique(s$dataset), m)
    if (is.null(p)) return(NULL)
    d <- copy(p$dot$data)
    d[, mechanism:=comparison_id]
    d
  }), fill=TRUE)
})
shared_effects <- rbindlist(effect_rows[2:3])
shared_size_values <- -log10(pmax(shared_effects$ora_fdr,1e-300))
shared_size_limits <- c(0,max(shared_size_values))
shared_size_breaks <- c(0,25,50)
shared_size_breaks <- shared_size_breaks[shared_size_breaks<=shared_size_limits[2]]
shared_fill_limits <- setNames(vapply(c("ARCHS4","local"),function(m) {
  z <- shared_effects[model==m]$logFC
  max(abs(scales::breaks_pretty(n=3)(range(c(-abs(z),abs(z))))))
},numeric(1)),c("ARCHS4","local"))
shared_dot_legends <- list()
dot_group <- function(i) {
  s <- copy(S[[i]])
  h <- effect_rows[[i]]
  order <- unique(s$comparison_id)
  levels_y <- rev(order)
  h[, mechanism:=factor(mechanism, levels=levels_y)]
  size_limits <- shared_size_limits
  # This is intentionally the Figure 3 G mechanism-recovery construction:
  # ARCHS4 at left, local at right, wide ARCHS4 field, independent fill keys
  # and one shared ORA-FDR size key.
  dot_one <- function(model_name, show_y) {
    d <- h[model==model_name]
    miss <- s[model==model_name & !recovered]
    miss[, mechanism:=factor(comparison_id, levels=levels_y)]
    contrasts <- unique(as.character(h$contrast))
    stopifnot(length(contrasts)==1L)
    x_missing <- contrasts[1]
    lim <- shared_fill_limits[[model_name]]
    ggplot(d, aes(contrast, mechanism)) +
      geom_point(aes(size=-log10(pmax(ora_fdr,1e-300)), fill=logFC), shape=21, stroke=.2) +
      geom_text(data=miss, aes(x=x_missing, y=mechanism), label="Not recovered",
                inherit.aes=FALSE, size=5/.pt, colour="grey35", fontface="italic") +
      scale_size_continuous(range=c(1,3.2), limits=size_limits, name="−log10 ORA FDR",
                            breaks=shared_size_breaks) +
      scale_fill_gradient2(low="#1a9850", mid="white", high="#d73027", midpoint=0,
                           limits=c(-lim,lim), name="GM log2FC") +
      scale_x_discrete(limits=contrasts, labels=function(x) gsub("_", " ", sub("tp_", "", x)), drop=FALSE) +
      scale_y_discrete(drop=FALSE) + labs(title=if(model_name=="local") proj_local_model_label(unique(s$dataset)) else "ARCHS4 projection", x=NULL, y=NULL) +
      nm_theme() + theme(axis.text.y=if(show_y) element_text(size=5) else element_blank(),
                         axis.ticks.y=if(show_y) element_line() else element_blank(),
                         axis.text.x=element_text(size=5, angle=0, hjust=.5), plot.title=element_text(size=6,hjust=.5),
                         legend.position="bottom", legend.box="vertical", legend.key.width=unit(3,"mm"),legend.key.height=unit(1.2,"mm"),
                         plot.margin=margin(1,1,1,1,"mm"))
  }
  arch <- dot_one("ARCHS4", FALSE); local <- dot_one("local", FALSE)
  if(i==2L) {
    shared_dot_legends$size <<- get_legend(arch + guides(fill="none",size=guide_legend(nrow=1,override.aes=list(fill="white"))) + theme(legend.position="bottom"))
    shared_dot_legends$arch <<- get_legend(arch + labs(fill="ARCHS4 GM log2FC") + guides(size="none",fill=guide_colourbar(title="ARCHS4 GM log2FC",title.position="top",title.hjust=.5,barwidth=unit(18,"mm"),barheight=unit(1.2,"mm"))) + theme(legend.position="bottom"))
    shared_dot_legends$trained <<- get_legend(local + labs(fill="Dataset-trained GM log2FC") + guides(size="none",fill=guide_colourbar(title="Dataset-trained GM log2FC",title.position="top",title.hjust=.5,barwidth=unit(18,"mm"),barheight=unit(1.2,"mm"))) + theme(legend.position="bottom"))
  }
  local <- local + theme(legend.position="none"); arch <- arch + theme(legend.position="none")
  list(arch=arch, local=local)
}
# Cytokine recovery is already shown in Fig. 3 and is therefore omitted here.
notebook_summary_table <- function(i) {
  d <- copy(S[[i]])
  d[, fraction:=fifelse(recovered & n_gene_set_in_universe > 0,
                        n_gene_set_in_top_loading / n_gene_set_in_universe, NA_real_)]
  d[, label:=fifelse(recovered, sprintf("%d/%d (%.0f%%)", n_gene_set_in_top_loading, n_gene_set_in_universe, 100*n_gene_set_in_top_loading/n_gene_set_in_universe), "—")]
  local_label <- proj_local_model_label(unique(d$dataset))
  d[, model_label:=factor(fifelse(model == "ARCHS4", "ARCHS4", local_label), levels=c("ARCHS4", local_label))]
  d[, mechanism:=factor(comparison_id, levels=rev(unique(comparison_id)))]
  ggplot(d, aes(model_label, mechanism, fill=fraction)) +
    geom_tile(colour="white", linewidth=.2) +
    geom_text(aes(label=label), size=5/.pt) +
    scale_fill_gradient(low="white", high="#1B9E77", limits=c(0,1), na.value="grey92",
                        name="Gene-set genes in top 1%\n(fraction)", breaks=c(0,.5,1)) +
    labs(x=NULL, y=NULL, title=NULL) + nm_theme() +
    theme(axis.text.x=element_text(size=5, angle=0, hjust=.5), axis.text.y=element_text(size=5),
          plot.title=element_text(size=5.5, hjust=.5), legend.position="bottom",
          legend.key.width=unit(8,"mm"), legend.key.height=unit(1.0,"mm"),
          plot.margin=margin(1,1,1,1,"mm"))
}
# B and C share one patchwork grid so the tile table and both dot plots use the same
# mechanism rows, and so b and c line up column by column. Each colour bar sits centred
# under its own column and the size key is centred across the two dot panels.
fraction_legend <- get_legend(notebook_summary_table(2)+guides(fill=guide_colourbar(title.position="top",title.hjust=.5,barwidth=unit(18,"mm"),barheight=unit(1.2,"mm")))+theme(legend.position="bottom"))
tag_theme <- theme(plot.tag=element_text(size=FS_TAG,face="bold",family="Helvetica"), plot.tag.position=c(.005,.995))
tile_B <- notebook_summary_table(2)+theme(legend.position="none")+labs(tag="c")+tag_theme
tile_C <- notebook_summary_table(3)+theme(legend.position="none")+labs(tag="d")+tag_theme
dots_B <- dot_group(2); dots_C <- dot_group(3)
n_B <- uniqueN(S[[2]]$comparison_id); n_C <- uniqueN(S[[3]]$comparison_id)
BC <- patchwork::wrap_plots(
  tile_B, dots_B$arch, dots_B$local,
  tile_C, dots_C$arch, dots_C$local,
  patchwork::wrap_elements(panel=fraction_legend), patchwork::wrap_elements(panel=shared_dot_legends$arch),
  patchwork::wrap_elements(panel=shared_dot_legends$trained),
  patchwork::plot_spacer(), patchwork::wrap_elements(panel=shared_dot_legends$size),
  design="ABC\nDEF\nGHI\nJKK", widths=c(1.25,1,1), heights=c(n_B, n_C, 2.4, 1.5))
BC <- patchwork::patchworkGrob(BC)

# Select two examples per model group. Prefer ARCHS4-only recoveries, then the
# largest ARCHS4 advantage in recovered pathway members.
example_candidates <- rbindlist(lapply(seq_along(S), function(g) {
  wide <- dcast(S[[g]], comparison_id + category ~ model,
    value.var=c("recovered", "LV", "gene_set", "fdr", "n_gene_set_in_top_loading", "n_gene_set_in_universe"))
  wide[, advantage:=n_gene_set_in_top_loading_ARCHS4 - n_gene_set_in_top_loading_local]
  only_arch <- wide[recovered_ARCHS4 == TRUE & recovered_local == FALSE][order(-n_gene_set_in_top_loading_ARCHS4)]
  # Show one clear ARCHS4-only recovery and one fair head-to-head example:
  # both models recover it, but ARCHS4 has more top-loading pathway genes and
  # a significant ARCHS4 FDR.
  both <- wide[recovered_ARCHS4 == TRUE & recovered_local == TRUE &
                 advantage > 0 & fdr_ARCHS4 <= .05][order(-advantage, fdr_ARCHS4)]
  chosen <- rbind(only_arch[seq_len(min(1L, nrow(only_arch)))],
                  both[seq_len(min(1L, nrow(both)))])
  if (nrow(chosen) < 2L) {
    fallback <- wide[recovered_ARCHS4 == TRUE & recovered_local == TRUE][order(-advantage, fdr_ARCHS4)]
    chosen <- unique(rbind(chosen, fallback))[seq_len(min(2L, nrow(unique(rbind(chosen, fallback)))))]
  }
  chosen[, group:=g]
  chosen
}), fill=TRUE)
# Two examples are sufficient at final size: one ARCHS4-only recovery, and one
# paired recovery in which ARCHS4 retains more pathway genes with significant FDR.
examples <- example_candidates
stopifnot(examples[, .N, by=group][, all(N == 2L)])

short_fdr <- function(x) ifelse(is.na(x), "—", formatC(x, format="e", digits=1))
# RNA-seq dataset (with accession) in which each example was found, shown under each title.
dataset_label <- c(cyt_ifna_GSE133218="IFN-alpha-treated EndoC-betaH1 cells (GSE133218)",
                   mono_lps_GSE193336="LPS-treated macrophages (GSE193336)",
                   placenta_EMTAB6701="Placental cell types (E-MTAB-6701)")
example_model_plot <- function(x, model_name, xmax) {
  g <- x$group; mechanism_name <- x$comparison_id
  s <- copy(S[[g]][comparison_id == mechanism_name])
  d <- copy(L[[g]][comparison_id == mechanism_name & model == model_name])
  st <- s[model == model_name]
  gene_labels <- d[is_gene_set == TRUE][order(rank)][seq_len(min(3L, .N))]
  ggplot(d, aes(rank, loading)) +
    geom_point(colour="grey78", size=.18, alpha=.35) +
    geom_point(data=d[is_gene_set==TRUE], colour=if (model_name == "ARCHS4") "#332288" else "#DDCC77", size=.55) +
    geom_text_repel(data=gene_labels, aes(label=gene), size=5/.pt, min.segment.length=0,
                    seed=1, max.overlaps=Inf, box.padding=.08, point.padding=.03) +
    scale_y_continuous(expand=expansion(mult=c(.03,.30))) +
    scale_x_continuous(limits=c(0, xmax)) +
    labs(x=NULL, y="Loading") + nm_theme() +
    theme(axis.text=element_text(size=5), axis.title.y=element_text(size=5), plot.margin=margin(1,2,1,2,"mm"))
}
example_model_header <- function(x, model_name) {
  s <- S[[x$group]][comparison_id == x$comparison_id]
  st <- s[model == model_name]
  model_label <- if (model_name == "ARCHS4") "ARCHS4 projection" else proj_local_model_label(unique(s$dataset))
  h <- ggdraw() + draw_label(model_label, y=.8, vjust=0, size=5.5, fontface="bold")
  if (!st$recovered[1]) return(h + draw_label("Not recovered", y=.45, vjust=0, size=5))
  e <- floor(log10(st$fdr[1])); m <- sprintf("%.1f", st$fdr[1] / 10^e)
  h + draw_label(sprintf("%s, %d/%d genes (%.0f%%)", st$LV[1], st$n_gene_set_in_top_loading[1], st$n_gene_set_in_universe[1], 100*st$n_gene_set_in_top_loading[1]/st$n_gene_set_in_universe[1]), y=.45, vjust=0, size=5) +
    draw_label(bquote("FDR" == .(m) %*% 10^displaystyle(.(e))), y=.06, vjust=0, size=5)
}
example_pair <- function(x) {
  g <- x$group; mechanism_name <- x$comparison_id
  s <- S[[g]][comparison_id == mechanism_name & model == "ARCHS4"]
  term <- proj_pretty_term(s$gene_set[1])
  # Each example is a direct side-by-side comparison at the same scale:
  # ARCHS4 | local.
  source_line <- dataset_label[[s$dataset[1]]]
  xmax <- max(L[[g]][comparison_id == mechanism_name, rank])
  plot_grid(ggdraw() + draw_label(stringr::str_wrap(term, width=42), y=.45, vjust=0, size=6) +
              draw_label(source_line, y=.12, vjust=0, size=5, colour="black"),
            plot_grid(example_model_header(x, "ARCHS4"), example_model_header(x, "local"), nrow=1),
            plot_grid(example_model_plot(x, "ARCHS4", xmax), example_model_plot(x, "local", xmax), nrow=1, align="hv", axis="tblr"),
            ggdraw() + draw_label("Gene rank", size=5),
            ncol=1, rel_heights=c(.19,.19,.56,.06))
}
E <- plot_grid(plotlist=lapply(seq_len(nrow(examples)), function(i) example_pair(examples[i])), ncol=3)

aligned <- cowplot::align_plots(A$pseudobulk, A$gtex, align="h", axis="tb")
A_row <- plot_grid(nm_tag(aligned[[1]], "a"), nm_tag(aligned[[2]], "b"),
                   nrow=1, rel_widths=c(1,1.2))
page <- plot_grid(
  A_row,
  BC,
  nm_tag(E, "e"),
  ncol=1, rel_heights=c(72,98,86))
nm_export(list(page), 6, "Projection benchmarks, mechanism recovery and local-versus-ARCHS4 gene-loading examples")

Supplementary Figure 6: 1 pages, 180 x 247 mm
